# BNEPA Royalty Report Analysis

Source: `BNEPA_Royalty_Report_MAY2026_LV.xlsx` — monthly China royalty statements.

**Task 1:** Extract the distinct list of `Product Name` + `Item Number` across all monthly sheets.

In [1]:
import pandas as pd
from pathlib import Path

WORKBOOK = Path("BNEPA_Royalty_Report_MAY2026_LV.xlsx")
SKIP_SHEETS = {"New template", "銷售庫存統計總表"}
HEADER_SCAN_DEPTH = 12  # rows scanned for the 'Product Name' header

In [2]:
def find_header_row(path: Path, sheet: str, depth: int = HEADER_SCAN_DEPTH) -> int:
    """Return the 0-indexed row that contains the 'Product Name' header."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    for i in range(len(probe)):
        cells = {str(v).strip() for v in probe.iloc[i].tolist() if pd.notna(v)}
        if "Product Name" in cells:
            return i
    raise ValueError(f"No 'Product Name' header found in first {depth} rows of {sheet!r}")

xl = pd.ExcelFile(WORKBOOK, engine="openpyxl")
monthly = [s for s in xl.sheet_names if s not in SKIP_SHEETS]
frames = {}
for s in monthly:
    hdr = find_header_row(WORKBOOK, s)
    frames[s] = pd.read_excel(xl, sheet_name=s, header=hdr, engine="openpyxl")
    print(f"  {s}: header at row {hdr + 1}, {len(frames[s])} data rows")

print(f"\nLoaded {len(frames)} monthly sheets")

  8月9月: header at row 8, 120 data rows


  10月: header at row 8, 123 data rows


  11月: header at row 8, 234 data rows


  12月: header at row 8, 203 data rows


  1月: header at row 8, 177 data rows


...

  12月2025: header at row 7, 463 data rows


  1月2026: header at row 7, 446 data rows


  2月2026 : header at row 7, 508 data rows


  3月2026: header at row 7, 476 data rows


  4月2026: header at row 7, 450 data rows

Loaded 20 monthly sheets


In [3]:
parts = []
for name, df in frames.items():
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    if not pn or not inum:
        print(f"  [skip] {name}: missing columns ({list(df.columns)[:5]}…)")
        continue
    sub = df[[pn, inum]].rename(columns={pn: "Product Name", inum: "Item Number"})
    sub["sheet"] = name
    parts.append(sub)

combined = pd.concat(parts, ignore_index=True)

# Drop rows where either field is missing, then normalise whitespace.
combined = combined.dropna(subset=["Product Name", "Item Number"], how="any")
combined["Product Name"] = combined["Product Name"].astype(str).str.strip()
combined["Item Number"] = combined["Item Number"].astype(str).str.strip()
combined = combined[(combined["Product Name"] != "") & (combined["Item Number"] != "")]

distinct = (
    combined[["Product Name", "Item Number"]]
    .drop_duplicates()
    .sort_values(["Product Name", "Item Number"])
    .reset_index(drop=True)
)
print(f"{len(distinct)} distinct (Product Name, Item Number) pairs")
distinct

242 distinct (Product Name, Item Number) pairs


,Product Name,Item Number
0,.hack//G.U. Last Recode,E05367
1,ACE COMBAT 7: SKIES UNKNOWN,E03174
2,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445
3,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444
4,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445
...,...,...
237,Towa and the Guardians of the Sacred Tree,E06936
238,Towa and the Guardians of the Sacred Tree Delu...,E06936
239,Towa and the Guardians of the Sacred Tree Delu...,L00117
240,We Love Katamari REROLL+ Royal Reverie,E05071


In [4]:
out = Path("distinct_products.csv")
distinct.to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/distinct_products.csv


## Task 2 — SRP history per product

For each (Product Name, Item Number), list the SRP (CNY) values it had and the month range each one applied to. Consecutive months at the same SRP collapse into a single range; a gap in sheets (a month where the product didn't sell) breaks the run.

In [5]:
import re
import datetime as _dt

DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")

def find_sales_period(path: Path, sheet: str, depth: int = 12) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Return (start_date, end_date) for the sheet by scanning rows below 'Sales Period :'."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    dates: list[pd.Timestamp] = []
    started = False
    for i in range(len(probe)):
        for v in probe.iloc[i].tolist():
            if pd.isna(v):
                continue
            if isinstance(v, str) and "Sales Period" in v:
                started = True
                continue
            if not started:
                continue
            if isinstance(v, (pd.Timestamp, _dt.datetime, _dt.date)):
                dates.append(pd.Timestamp(v))
            elif isinstance(v, str) and DATE_RE.match(v):
                dates.append(pd.Timestamp(v))
            if len(dates) == 2:
                return dates[0], dates[1]
    raise ValueError(f"Could not find sales-period dates in {sheet!r}")

sheet_periods = {s: find_sales_period(WORKBOOK, s) for s in monthly}
sheet_order = sorted(sheet_periods, key=lambda s: sheet_periods[s][0])
for s in sheet_order:
    a, b = sheet_periods[s]
    print(f"  {s}: {a.date()} → {b.date()}")

  8月9月: 2024-08-01 → 2024-09-30
  10月: 2024-10-01 → 2024-10-31
  11月: 2024-11-01 → 2024-11-30
  12月: 2024-12-01 → 2024-12-31
  1月: 2025-01-01 → 2025-01-31
  2月: 2025-02-01 → 2025-02-28
  3月: 2025-03-01 → 2025-03-31
  4月: 2025-04-01 → 2025-04-30
  5月: 2025-05-01 → 2025-05-31
  6月: 2025-06-01 → 2025-06-30
  7月: 2025-07-01 → 2025-07-31
  8月: 2025-08-01 → 2025-08-30
  9月: 2025-09-01 → 2025-09-30
  10月2025: 2025-10-01 → 2025-10-31
  11月2025: 2025-11-01 → 2025-11-30
  12月2025: 2025-12-01 → 2025-12-31
  1月2026: 2026-01-01 → 2026-01-31
  2月2026 : 2026-02-01 → 2026-02-28
  3月2026: 2026-03-01 → 2026-03-31
  4月2026: 2026-04-01 → 2026-04-30


In [6]:
srp_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn, inum, srp = cols.get("Product Name"), cols.get("Item Number"), cols.get("SRP (CNY)")
    if not (pn and inum and srp):
        print(f"  [skip] {sheet}: missing one of Product Name / Item Number / SRP (CNY)")
        continue
    sub = df[[pn, inum, srp]].rename(
        columns={pn: "Product Name", inum: "Item Number", srp: "SRP (CNY)"}
    )
    sub = sub.dropna(subset=["Product Name", "Item Number", "SRP (CNY)"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub["SRP (CNY)"] = pd.to_numeric(sub["SRP (CNY)"], errors="coerce")
    sub = sub.dropna(subset=["SRP (CNY)"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    srp_parts.append(sub)

srp_long = pd.concat(srp_parts, ignore_index=True)

# Within a single sheet a product can appear multiple times (per Customer); keep the modal SRP.
srp_long = (
    srp_long.groupby(["Product Name", "Item Number", "sheet", "period_start", "period_end"], as_index=False)
    ["SRP (CNY)"].agg(lambda s: s.mode().iloc[0])
)
print(f"{len(srp_long)} (product, item, sheet) SRP observations")
srp_long.head()

2942 (product, item, sheet) SRP observations


,Product Name,Item Number,sheet,period_start,period_end,SRP (CNY)
0,.hack//G.U. Last Recode,E05367,10月,2024-10-01,2024-10-31,278.0
1,.hack//G.U. Last Recode,E05367,10月2025,2025-10-01,2025-10-31,278.0
2,.hack//G.U. Last Recode,E05367,11月,2024-11-01,2024-11-30,278.0
3,.hack//G.U. Last Recode,E05367,11月2025,2025-11-01,2025-11-30,278.0
4,.hack//G.U. Last Recode,E05367,12月,2024-12-01,2024-12-31,278.0


In [7]:
from pandas.tseries.offsets import DateOffset

# Sort each product's observations chronologically, then collapse consecutive same-SRP
# rows into ranges. A "gap" (sheet missing between two observations) breaks the run.
sheet_idx = {s: i for i, s in enumerate(sheet_order)}
srp_long["_idx"] = srp_long["sheet"].map(sheet_idx)
srp_long = srp_long.sort_values(["Product Name", "Item Number", "_idx"]).reset_index(drop=True)

runs = []
for (prod, item), grp in srp_long.groupby(["Product Name", "Item Number"], sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    run_srp = None
    prev_idx = None
    for _, row in grp.iterrows():
        if run_srp is None:
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        same_srp = row["SRP (CNY)"] == run_srp
        contiguous = row["_idx"] == prev_idx + 1
        if same_srp and contiguous:
            run_end = row["period_end"]
        else:
            runs.append((prod, item, run_srp, run_start, run_end))
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if run_srp is not None:
        runs.append((prod, item, run_srp, run_start, run_end))

srp_history = pd.DataFrame(
    runs, columns=["Product Name", "Item Number", "SRP (CNY)", "start_date", "end_date"]
)
srp_history["start_month"] = srp_history["start_date"].dt.strftime("%Y-%m")
srp_history["end_month"] = srp_history["end_date"].dt.strftime("%Y-%m")
srp_history = srp_history[
    ["Product Name", "Item Number", "SRP (CNY)", "start_month", "end_month", "start_date", "end_date"]
].sort_values(["Product Name", "Item Number", "start_date"]).reset_index(drop=True)

print(f"{len(srp_history)} SRP runs across {srp_history[['Product Name','Item Number']].drop_duplicates().shape[0]} products")
srp_history.head(20)

423 SRP runs across 242 products


,Product Name,Item Number,SRP (CNY),start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,E05367,278.0,2024-08,2025-04,2024-08-01,2025-04-30
1,.hack//G.U. Last Recode,E05367,278.0,2025-06,2026-04,2025-06-01,2026-04-30
2,ACE COMBAT 7: SKIES UNKNOWN,E03174,268.0,2024-08,2024-10,2024-08-01,2024-10-31
3,ACE COMBAT 7: SKIES UNKNOWN,E03174,268.0,2024-12,2026-04,2024-12-01,2026-04-30
4,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445,368.0,2024-08,2024-10,2024-08-01,2024-10-31
5,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445,368.0,2025-10,2025-12,2025-10-01,2025-12-31
6,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445,368.0,2026-02,2026-03,2026-02-01,2026-03-31
7,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444,498.0,2024-08,2024-10,2024-08-01,2024-10-31
8,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444,498.0,2025-10,2026-04,2025-10-01,2026-04-30
9,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445,368.0,2025-01,2026-04,2025-01-01,2026-04-30


In [8]:
out = Path("product_srp_history.csv")
srp_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_srp_history.csv


## Task 3 — Promotion history per (product, customer)

For each (Product Name, Item Number, Customer, Promo Discount), list the month range each promo was active. Customer is from the newer-layout sheets; the older sheets (8月9月 → 6月) have no Customer column and are labelled `All`. Rows where `Promo Discount = 0` (no promo) and aggregate rows (`SUBTOTAL`, `TOTAL`) are excluded.

In [9]:
PROMO_COL_VARIANTS = ("Promo Discount\n(OFF)", "Promo Discount (OFF)")
AGGREGATE_CUSTOMERS = {"SUBTOTAL", "TOTAL"}

promo_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    promo = next((cols[k] for k in PROMO_COL_VARIANTS if k in cols), None)
    cust = cols.get("Customer")
    if not (pn and inum and promo):
        print(f"  [skip] {sheet}: missing Product Name / Item Number / Promo Discount")
        continue
    keep = [pn, inum, promo] + ([cust] if cust else [])
    sub = df[keep].copy()
    rename = {pn: "Product Name", inum: "Item Number", promo: "Promo Discount"}
    if cust:
        rename[cust] = "Customer"
    sub = sub.rename(columns=rename)
    if "Customer" not in sub.columns:
        sub["Customer"] = "All"

    sub = sub.dropna(subset=["Product Name", "Item Number", "Promo Discount"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub["Customer"] = sub["Customer"].fillna("All").astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub = sub[~sub["Customer"].str.upper().isin(AGGREGATE_CUSTOMERS)]
    sub["Promo Discount"] = pd.to_numeric(sub["Promo Discount"], errors="coerce")
    sub = sub.dropna(subset=["Promo Discount"])
    sub = sub[sub["Promo Discount"] > 0]
    if sub.empty:
        continue
    sub = sub.drop_duplicates(subset=["Product Name", "Item Number", "Customer", "Promo Discount"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    promo_parts.append(sub)

promo_long = pd.concat(promo_parts, ignore_index=True)
print(f"{len(promo_long)} (product, item, customer, promo, sheet) observations")
print("customers seen:", sorted(promo_long["Customer"].unique()))
promo_long.head()

3022 (product, item, customer, promo, sheet) observations
customers seen: ['All', 'Heybox', 'Sonkwo']


,Product Name,Item Number,Promo Discount,Customer,sheet,period_start,period_end
0,SPY×ANYA: Operation Memories,E05697,0.3000,All,11月,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,E05367,0.6835,All,11月,2024-11-01,2024-11-30
2,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6800,All,11月,2024-11-01,2024-11-30
3,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6835,All,11月,2024-11-01,2024-11-30
4,Ace Combat 7: Skies Unknown,E03174,0.6716,All,11月,2024-11-01,2024-11-30


In [10]:
promo_long["_idx"] = promo_long["sheet"].map(sheet_idx)
promo_long = promo_long.sort_values(
    ["Product Name", "Item Number", "Customer", "Promo Discount", "_idx"]
).reset_index(drop=True)

promo_runs = []
group_keys = ["Product Name", "Item Number", "Customer", "Promo Discount"]
for (prod, item, customer, promo), grp in promo_long.groupby(group_keys, sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    prev_idx = None
    for _, row in grp.iterrows():
        if prev_idx is None:
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        if row["_idx"] == prev_idx + 1:
            run_end = row["period_end"]
        else:
            promo_runs.append((prod, item, customer, promo, run_start, run_end))
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if prev_idx is not None:
        promo_runs.append((prod, item, customer, promo, run_start, run_end))

promo_history = pd.DataFrame(
    promo_runs,
    columns=["Product Name", "Item Number", "Customer", "Promo Discount", "start_date", "end_date"],
)
promo_history["start_month"] = promo_history["start_date"].dt.strftime("%Y-%m")
promo_history["end_month"] = promo_history["end_date"].dt.strftime("%Y-%m")
promo_history = promo_history[
    ["Product Name", "Item Number", "Customer", "Promo Discount",
     "start_month", "end_month", "start_date", "end_date"]
].sort_values(["Product Name", "Item Number", "Customer", "start_date", "Promo Discount"]).reset_index(drop=True)

print(f"{len(promo_history)} promo runs")
print(f"  across {promo_history[['Product Name','Item Number']].drop_duplicates().shape[0]} products")
print(f"  across {promo_history[['Product Name','Item Number','Customer']].drop_duplicates().shape[0]} (product, customer) pairs")

out = Path("product_promo_history.csv")
promo_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")
promo_history.head(20)

1304 promo runs
  across 221 products
  across 504 (product, customer) pairs
Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_promo_history.csv


,Product Name,Item Number,Customer,Promo Discount,start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,E05367,All,0.6835,2024-11,2024-11,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,E05367,All,0.9000,2024-12,2025-03,2024-12-01,2025-03-31
2,.hack//G.U. Last Recode,E05367,All,0.8000,2025-04,2025-04,2025-04-01,2025-04-30
3,.hack//G.U. Last Recode,E05367,All,0.9000,2025-06,2025-06,2025-06-01,2025-06-30
4,.hack//G.U. Last Recode,E05367,Heybox,0.9000,2025-07,2025-09,2025-07-01,2025-09-30
5,.hack//G.U. Last Recode,E05367,Heybox,0.8000,2025-10,2026-04,2025-10-01,2026-04-30
6,.hack//G.U. Last Recode,E05367,Heybox,0.9000,2026-04,2026-04,2026-04-01,2026-04-30
7,.hack//G.U. Last Recode,E05367,Sonkwo,0.9000,2025-07,2025-07,2025-07-01,2025-07-31
8,.hack//G.U. Last Recode,E05367,Sonkwo,0.8000,2025-11,2026-02,2025-11-01,2026-02-28
9,ACE COMBAT 7: SKIES UNKNOWN,E03174,All,0.8000,2024-12,2025-05,2024-12-01,2025-05-31


## Task 4 — Resolve Playasia PAX codes by name

`lookup_pax_codes.py` queries the Playasia `/products/filtered` endpoint (type=`digital codes`, digital=1) for each unique Product Name and writes `distinct_products_with_pax.csv` with the top match. This cell loads that output and surfaces rows that need attention: ones with no result, ones where the CSV's existing Item Number differs from the Playasia PAX code, and the original `0` Item Number rows.

In [ ]:
pax = pd.read_csv('distinct_products_with_pax.csv', dtype=str).fillna('')
pax['lookup_score'] = pd.to_numeric(pax['lookup_score'], errors='coerce')
print('match_status counts:')
print(pax['match_status'].value_counts().to_string())
print(f"\nmean lookup_score: {pax['lookup_score'].mean():.1f}")
pax.head()

### Rows with no Playasia match

In [ ]:
pax[pax['match_status'] == 'not_found'][['Product Name', 'Item Number']]

### Rows where the CSV's Item Number was missing (`0`) — recovered PAX codes

In [ ]:
pax[pax['match_status'] == 'csv_invalid'][['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score']]

### Lowest-confidence "differs" rows (sorted by fuzzy score — review these manually)

In [ ]:
(pax[pax['match_status'] == 'differs']
 .sort_values('lookup_score')
 [['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score', 'lookup_total_count']]
 .head(30))